# 멀티모달 RAG
- 동영상 검색 실습

## 0.환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = "/content/drive/MyDrive"

In [ ]:
%pip install --upgrade openai pandas tqdm pinecone langchain langchain-openai langchain-pinecone langchain-text-splitters pillow

- 라이브러리 불러오기

In [ ]:
import os
import uuid
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from PIL import Image
import base64
from IPython.display import display, HTML
from google.colab import userdata

- Key 설정

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

OPENAI_LLM_MODEL = userdata.get("OPENAI_LLM_MODEL")
OPENAI_EMBEDDING_MODEL = userdata.get("OPENAI_EMBEDDING_MODEL")


PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
PINECONE_INDEX_REGION = userdata.get("PINECONE_INDEX_REGION")
PINECONE_INDEX_CLOUD = userdata.get("PINECONE_INDEX_CLOUD")
PINECONE_INDEX_NAME = userdata.get("PINECONE_INDEX_NAME")
PINECONE_INDEX_METRIC = userdata.get("PINECONE_INDEX_METRIC")
PINECONE_INDEX_DIMENSION = int(userdata.get("PINECONE_INDEX_DIMENSION"))

## 1.데이터 전처리
- 데이터 수집 -> 데이터 병합



- 1.데이터 수집

In [ ]:
comments_dir = 
comments_dir_path = 

csv_files = [

]

print(csv_files)  # 파일 경로가 잘 나오면 OK

- 2.데이터 병합

In [ ]:
dfs = []

for f in csv_files:
    try:
        df = pd.read_csv(f)
        print(f"{os.path.basename(f)} shape: {df.shape}")

        if not df.empty:
            dfs.append(df)
    except Exception as e:
        # 파일 읽기 중 에러 발생 시 파일명과 에러 메시지 출력
        print(f"Error reading {f}: {e}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(df_all.shape)
    display(df_all)
else:
    print("⚠️ 읽을 데이터가 없습니다.")           # 파일이 없거나 비어있을 경우


## 2.텍스트 청킹(Chunking)
- `df_all`에 통합된 모든 댓글 데이터를 벡터 임베딩하기 전에 **LangChain의 RecursiveCharacterTextSplitter** 를 사용해 텍스트를 일정 길이로 나누는 단계
- 벡터 저장소 구축의 핵심 전처리 구간

In [ ]:
text_splitter = 

chunks = []

for idx, row in tqdm(df_all.iterrows(), total=len(df_all), desc="청킹 중"):
    for chunk in text_splitter.split_text(str(row["description"])):
        chunks.append({
            "id": str(uuid.uuid4()),
            "comment": ,
            "frame_no": row["frame_no"],
            "video_filename": row["video_filename"]
        })
df_chunks =


## 3.벡터 저장소(Vector Store) 생성
- 벡터를 저장할 인덱스 생성 및 연결

In [ ]:
if PINECONE_INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=,
        dimension=,
        metric=,
        spec=ServerlessSpec(
            cloud=,
            region=
        )
    )
index = pc.Index()
index.describe_index_stats()

## 4.임베딩(Embedding)

In [ ]:

embeddings_model = 


vectorstore = 


texts = 

metadatas = [
    {
        "id": ,
        "frame_no": ,
        "video_filename": 
    }
    for _, row in df_chunks.iterrows()
]


batch_size = 

for i in tqdm(range(0, len(texts), batch_size), desc="Pinecone 업서트 중"):
    batch_texts = texts[i:i+batch_size]
    batch_metadatas = metadatas[i:i+batch_size]
    vectorstore.add_texts(
        texts=batch_texts,
        metadatas=batch_metadatas
    )

print(f"\n\nPinecone 인덱스({PINECONE_INDEX_NAME})에 총 {len(texts)}개 벡터 업서트 완료.")

## 5.데이터 확인

In [ ]:
query = " "


top_k = 
results = 

In [ ]:

def show_frame_image_from_metadata(doc, frames_dir="frames", base_path='.', thumb_size=(240, 135)):
    meta = doc.metadata
    video_filename = meta["video_filename"]
    frame_no = int(meta["frame_no"])
    video_basename = os.path.splitext(video_filename)[0]
    image_filename = f"{video_basename}_frame{frame_no:05d}.jpg"
    image_path = os.path.join(base_path, frames_dir, video_basename, image_filename)
    if not os.path.exists(image_path):
        print(f"파일이 없습니다: {image_path}")
        return
    image = Image.open(image_path)
    image.thumbnail(thumb_size)  # 종횡비 유지, thumb_size 이내로 축소
    display(image)


In [ ]:
for i, (doc, score) in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print("텍스트:", )
    print("메타데이터:", )
    print("유사도 점수:", )
    show_frame_image_from_metadata(doc, frames_dir="frames", base_path=base_path, thumb_size=(240,135))
    print()

## 6.RAG 체인 구성

In [ ]:
llm = 

In [ ]:
prompt = 



def build_context(docs):
    """
    docs: List[Document]  # LangChain 문서 객체
    return: str           # LLM에 넣을 컨텍스트 텍스트
    """
    context = ""
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        # video_filename, frame_no가 없는 경우 대비해 get() 사용
        context += (
            f"[{i}]\n"
            f"비디오: {m.get('video_filename')}, 프레임: {m.get('frame_no')}\n"
            f"해설: {doc.page_content}\n\n"
        )
    return context


def get_frame_image_path(video_filename, frame_no, frames_dir="frames", base_path="."):
    video_basename = os.path.splitext(video_filename)[0]
    fname = f"{video_basename}_frame{int(frame_no):05d}.jpg"
    return os.path.join(base_path, frames_dir, video_basename, fname)



def display_video_file_base64(video_filename, videos_dir="videos", base_path=".", width=480):
    video_path = os.path.join(base_path, videos_dir, video_filename)
    if not os.path.exists(video_path):
        print(f"비디오 파일이 존재하지 않습니다: {video_path}")
        return
    with open(video_path, "rb") as f:
        video_bytes = f.read()
    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
    video_tag = f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
      Your browser does not support the video tag.
    </video>
    """
    display(HTML(video_tag))



def display_frame_image(frame_img_path, width=480):
    if not os.path.exists(frame_img_path):
        print(f"이미지 파일이 존재하지 않습니다: {frame_img_path}")
        return
    with open(frame_img_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    display(HTML(f'<img src="data:image/jpeg;base64,{data}" width="{width}"/>'))

## 7.검색 및 응답
- 벡터 검색 → 컨텍스트 구성 → LLM 답변 생성 → 프레임 정보 추출

In [ ]:
def search_and_answer(vectorstore, llm, query, k=3, base_path="."):

    docs_and_scores =
    docs = [d for d, score in docs_and_scores]
    if not docs:
        return {
            "answer": "검색된 관련 프레임이 없습니다.",
            "video_filename": None,
            "frame_no": None,
            "frame_img_path": None
        }


    context = 


    messages = 

    response = 


    meta = 
    video_filename = meta["video_filename"]
    frame_no = meta["frame_no"]
    frame_img_path = get_frame_image_path(video_filename, frame_no, frames_dir="frames", base_path=base_path)
    return {
        "answer": ,
        "video_filename": ,
        "frame_no": ,
        "frame_img_path": 
    }

In [ ]:
query = 

In [ ]:
result = 

print("===== RAG 답변 =====")
print(result["answer"])
if result["video_filename"] is not None:
    print(f"\n비디오 파일: {result['video_filename']}")
    print(f"프레임 번호: {result['frame_no']}")
    print("\n비디오 플레이어:")
    display_video_file_base64(result['video_filename'], base_path=base_path)
    print("관련 프레임 이미지:")
    display_frame_image(result['frame_img_path'])
else:
    print("추천할 비디오/프레임 정보가 없습니다.")